# Streamlit — From AI Prototype to Product

<div class="slide-title">

# Streamlit — From AI Prototype to Product
    
</div>

Notes: This lecture frames Streamlit from a product management perspective — understanding the mental model, the trade-offs, and how it fits into an AI product workflow. Technical depth is kept where it helps PMs reason about scope, risk, and stakeholder communication. **Feel free to bring in additional examples from products your cohort knows.**

### Warm-up
Go to [streamlit.io/gallery](https://streamlit.io/gallery) and explore a few example apps for 3 minutes. Discuss:

+ What kind of product decision could a stakeholder make using one of these apps?
+ How is this different from receiving a static slide deck or PDF report?
+ Where would this fit in your product development cycle — early validation, internal tooling, or something else?
+ If you were a PM, what would you want to test with a Streamlit prototype before commissioning a full build?


## Motivation

### The last mile problem

<div class="group">
    <div class="text_70">

* AI development has a _last mile problem_
    * Models, analyses, and prototypes live in Python — technical environments
    * The people who need to act on the AI's output do not live in that world
    * As an AI PM, you sit exactly at this gap
        
    </div>
    <div class="images_30">
    </div>
</div>

The last mile is a concept from logistics — getting the product the final stretch to the customer is often the hardest and most expensive part. In AI, the product is a model or an insight, and the last mile is getting it in front of decision-makers in a form they can actually use. As an AI PM, closing that gap is often your core job.

* Traditionally, two unsatisfying options for sharing AI work:
    * **Static reports** (PDFs, slide decks) — capture a snapshot but lose interactivity; stakeholders can't explore the model themselves
    * **Commission a full product** — preserves interactivity but takes months of engineering time before you can validate whether the idea is even worth building

* The fundamental tension in AI product development:
    * Engineers and data scientists think in **Python and models**
    * Stakeholders think in **interfaces and decisions**
    * PMs are responsible for bridging that gap — cheaply and quickly

Streamlit resolves this tension by turning Python analysis directly into a shareable interface. For a PM, this means you can put a working AI prototype in front of stakeholders in hours — not months — and collect real feedback before any engineering investment in a production system.

### What Streamlit is — and is not

* **Is**: The shortest path from a working Python or AI prototype to a shareable, interactive tool
* **Is not**: A general-purpose web framework — it has no routing, authentication, or user management built in

* It is a niche tool for clear purposes:
    * **Internal tools** — let non-technical colleagues query a model or filter data themselves
    * **Stakeholder demos** — show a working AI feature before committing to a full build
    * **Validation experiments** — test whether a PM hypothesis holds against real model behavior
    * **Rapid prototyping** — go from idea to clickable demo in hours, not sprints

As a PM, knowing these boundaries is critical. Promising a Streamlit prototype to an enterprise client as a production system is a scope mistake. It is excellent for validating ideas and communicating with stakeholders. It is not a substitute for a properly engineered product.

### Three design principles

Streamlit rests on three principles that shape everything about the framework:

1. **An app is a script, not an architecture**
    * No classes to inherit, no routes to define, no templates to write


2. **Interaction = re-execution**
    * When a user changes an input, the script reruns from the top
    * No callbacks, no event listeners, no wiring

3. **The API should disappear**
    * So small and intuitive that it never becomes the focus
    * Your analysis stays front and center

These three principles are deliberate product decisions by Streamlit's creators. Each one trades something — flexibility, fine-grained control, feature richness — for radical simplicity. Understanding these trade-offs helps you set correct expectations with your engineering team and your stakeholders.

## How Streamlit Works: The Mental Model

### Scripts, not applications

<div class="group">
    <div class="text_70">

* In traditional web development, an app is a complex **architecture**
    * Routes, request handlers, frontend components, API layers
    * Building one typically takes a team and multiple sprints
        
    </div>
    <div class="images_30">
    </div>
</div>

* Streamlit **inverts** this entirely
    * A Streamlit app is just a Python script that runs from top to bottom
    * Each line that produces output appears in the browser — in order
    * **No frontend/backend split, no API design, no routing decisions**

As a PM, this matters because it directly affects scope and velocity. The script IS the app. There is no separate frontend to build, no API to design. This is why a data scientist or AI engineer can build a functional prototype in a day — and why you should not be quoting web development timelines for Streamlit work.

### Reactivity: how interaction works

<div class="group">
    <div class="text_70">

* When a user interacts with a widget (slider, dropdown, button), Streamlit does something simple:
    * It **reruns the entire script** from the beginning
    * The widget remembers its new value
    * Everything downstream recomputes automatically
        
    </div>
    <div class="images_30">
    </div>
</div>

* This is the simplest possible reactivity model
    * Mental model: "changing an input re-generates the entire page"
    * No callbacks, no complex event wiring — it just reruns
    * **PM implication**: interactions feel immediate, but performance depends on how fast the script runs

Think of this like a spreadsheet: when you change a cell value, all the dependent cells recalculate immediately. Streamlit applies the same idea to an entire Python application. The user changes an input, and the whole "spreadsheet" recalculates.

### The performance trade-off

* The simplicity of full re-execution has a direct cost: **speed**
    * If loading a model takes 10 seconds, every interaction takes 10 seconds
    * This is acceptable for demos; it is not acceptable for real users

* Streamlit solves this with **caching** — which we will discuss shortly
    * For a PM: caching is not optional for AI apps; ask your team whether it is in place

* This performance ceiling is also a **product signal**:
    * If the demo feels slow even with caching, it may indicate the underlying model is too slow for a real product
    * Streamlit's slowness can surface latency problems early — before you've invested in a full build

This is a PM-relevant insight. If your team's Streamlit demo of an LLM feature takes 15 seconds to respond to every prompt, that's not just a Streamlit problem — it may be a sign the model is too slow for real users. Prototypes expose real constraints.

## The Interface Model

### Declaring what to show, not how to build it

<div class="group">
    <div class="text_70">

* Traditional web development is **imperative**: you specify every step of construction
    * Create a component, position it, register an event listener, update on change
* Streamlit is **declarative**: you specify *what* should appear
    * `st.slider(...)` → renders a slider, returns its value
    * `st.dataframe(df)` → renders an interactive table

    </div>
    <div class="images_30">
    </div>
</div>

Same distinction as SQL vs. writing database access code by hand.
SQL: "give me rows where churn_risk > 0.8" — the database figures out how.
Streamlit: "display this AI model output" — the framework handles the rendering.
For a PM: you specify the output you want, not the engineering steps to get there.

### Widgets as inputs

* In traditional development, a slider is a UI **component** requiring setup and event wiring
* In Streamlit, a slider is an **expression** that returns a value:
    * `confidence = st.slider("Confidence threshold", 0.0, 1.0, 0.5)` → the variable `confidence` holds whatever the user set

* A single line does three things simultaneously:
    1. **Renders** a slider in the browser
    2. **Returns** the current value to the Python logic
    3. **Triggers** a full re-execution when changed

* For a PM: widgets are the **parameters** you expose to stakeholders — they define what the user can control

* Choosing which widgets to expose is a **product decision**:
    * Too many parameters → overwhelming, unclear what to change
    * Too few → stakeholders cannot explore what they need
    * The PM's job: decide what knobs belong in the interface

Deciding which parameters to expose is exactly the same problem as product design: what does the user actually need to control? A churn model demo might expose "customer segment" and "date range" but hide the model hyperparameters. The PM defines the surface area of the prototype.

### Automatic rendering

* `st.write()` inspects the **type** of the object it receives and renders it appropriately:
    * String → Markdown text
    * DataFrame → interactive table
    * Chart figure → visual chart
    * Dictionary → formatted JSON
* **No configuration needed** — the framework chooses the right presentation

* For a PM: you can think of `st.write()` as "display this, whatever it is"
    * It lowers the cost of showing outputs to near-zero
    * This is a useful analogy for user-facing AI features too: smart defaults reduce friction for end users

## Layout: Organizing Information for Stakeholders

### How layout works

* Default layout is **vertical**: each output appears below the previous one
* Several mechanisms to organize information for users:
    * **Columns** — results side by side (comparisons, metrics)
    * **Sidebar** — persistent controls on the left (parameters the user adjusts)
    * **Tabs** — different views without scrolling (e.g., "Model Output" / "Explanation" / "Data")
    * **Expanders** — optional detail for advanced users

* The model is a **document with sections**, not a pixel-perfect canvas
    * Apps look clean and professional by default without any design work
    * Trade-off: no fine-grained visual control, but also no design debt to manage

## State: What Streamlit Remembers

### The sidebar pattern

* The most common and most effective layout in Streamlit mirrors professional dashboard tools:
    * **Controls** on the left — the parameters the user adjusts
    * **Results** in the center — what the AI produces given those parameters
* This is a strong default for AI product demos because it separates **questions** from **answers**

* This pattern directly maps to a PM concern: **information hierarchy**
    * Sidebar: "What parameters should the user be able to change?"
    * Main area: "What does the user ultimately need to see?"
    * Tabs: "Is there secondary information that some users need but others don't?"

Thinking through layout is a product exercise. An AI PM should be able to sketch: "this slider goes in the sidebar, the model predictions go in a center table, and we hide the confidence scores in an expander for analysts who want them." You don't need to code it — but you should be able to spec it.

### The stateless paradox

<div class="group">
    <div class="text_70">

* Streamlit's re-execution model is **stateless by design**
    * Every interaction reruns the script from scratch
* But real AI applications need **memory**:
    * A chat history that grows with each message
    * A multi-step configuration workflow
    * User selections that carry forward across steps
        
    </div>
    <div class="images_30">
    </div>
</div>

* Streamlit solves this with two mechanisms: **caching** and **session state**

### Caching: avoiding recomputation

<div class="group">
    <div class="text_70">

* `@st.cache_data` — for data and computations
    * Loading a dataset, running a prediction, calling an API
    * If the function is called again with the same inputs, it returns the stored result instead of rerunning
    * **PM implication**: this is what makes AI demos feel responsive even when models are expensive
        
    </div>
    <div class="images_30">
    </div>
</div>

* `@st.cache_resource` — for objects that are expensive to load once
    * An LLM, an embedding model, a database connection
    * Loaded once, shared across all users and interactions
    * **PM implication**: a model loaded into memory is available instantly to everyone using the app

### Session State: multi-step interactions

<div class="group">
    <div class="text_70">

* `st.session_state` is a dictionary that persists across reruns for each user
    * Enables conversational AI demos: the chat history lives here
    * Enables multi-step flows: a user's choices in step 1 are available in step 2
    * **Product use case**: any AI feature that requires context over multiple turns needs session state
        
    </div>
    <div class="images_30">
    </div>
</div>

* Together, **caching** and **session state** make Streamlit feel like a real product
    * Caching: "the model responds fast"
    * Session state: "the app remembers what happened before"

* The mental model for a PM:
    * `cache_data` → **speed**: avoids repeating expensive computation
    * `cache_resource` → **availability**: one model loaded, available to all users
    * `session_state` → **continuity**: the app remembers each user's journey

### Why this matters for AI PMs

* LLM-based features in particular are expensive and slow by default
* Without caching: every slider move re-calls the model → unusable demo
* Without session state: the chat loses context on every interaction → broken experience
* **As a PM**: when reviewing an AI Streamlit prototype, check that both are in place

Together, caching and session state create the illusion of a persistent, responsive product — even though the underlying mechanism is repeated full re-execution. From a product standpoint: this illusion is good enough for demos, stakeholder validation, and internal tools. Whether it is good enough for production depends on scale and user expectations.

A practical checkpoint: before showing a Streamlit prototype to stakeholders, ask your team: "Is the model cached? Does the chat history persist?" If not, the demo will feel broken — not because the AI is bad, but because the prototype mechanics are missing.

## From AI Feature to Stakeholder Demo

### The conceptual distance is small

<div class="group">
    <div class="text_70">

* A typical AI engineering workflow:
    * Experiment in a notebook → train a model → evaluate → iterate
    * Outputs are technical: metrics, plots, logged artifacts
* A Streamlit app takes that same workflow and wraps it in a usable interface:
    * Replace hardcoded inputs with user-controlled widgets
    * Replace internal `print()` and `display()` with `st.write()`
    * Cache the expensive steps once
        
    </div>
    <div class="images_30">
    </div>
</div>

* **No new model, no new architecture, no new backend**
    * The AI logic stays exactly the same
    * Streamlit adds a layer of interactivity on top of existing work
    * **PM implication**: asking your team for a Streamlit demo of an existing model is a small ask

Key PM mindset: "Can we wrap what already exists in a Streamlit interface so I can show it to the stakeholders this week?" That is almost always a reasonable ask, and almost always a faster path to feedback than any other option.

### When Streamlit is not enough

* Streamlit's simplicity has real limits — things that require a proper product build:
    * **User authentication** — who is allowed to see this?
    * **Persistent data storage** — users saving their work, audit trails, user profiles
    * **Complex navigation** — multiple pages, deep linking, mobile
    * **Fine-grained visual design** — custom branding, pixel-level layout control

* Streamlit is best used as a **validation tool**, not a production system:
    * Build the Streamlit demo to test the idea
    * Show it to stakeholders and collect real feedback
    * **Only** invest in a full product build (Flask, FastAPI + React, cloud product) if the idea proves its value
    * This is a lean product development principle applied to AI

Notes: This is one of the most important lessons for an AI PM. Knowing the limits of Streamlit — and being able to explain them to stakeholders — prevents scope creep and misaligned expectations. "We can have a demo in a week" and "we can have a production app in a week" are very different statements. Make sure stakeholders understand the distinction before they see the demo.

## Deployment

### Closing the communication gap

* An app on `localhost` is only useful to the person who built it
    * Deployment is what actually puts the prototype in stakeholders' hands

* **Streamlit Community Cloud**: the fastest path to a shareable link
    * Connect a GitHub repo with a `requirements.txt`
    * App is live at a public URL within minutes
    * Free for public repositories — ideal for demos and stakeholder reviews

* **Any Python host** works if you need more control:
    * Docker containers, cloud VMs, internal platforms
    * A Streamlit app is just a Python process — no build step, no compiled frontend, no separate API server
    * **PM implication**: your infrastructure team can deploy it on existing infrastructure without special tooling

## Conclusion

### Key Takeaways for AI PMs

* Streamlit solves the **last mile problem**: getting AI outputs from a Python prototype to stakeholders


* The **execution model** is a script that reruns on every interaction — simple by design, fast to build

* **Widgets** define what stakeholders can control — choosing them is a product decision, not just an engineering one

* **Caching** makes AI demos usable; **session state** makes them feel continuous — both are required for LLM features

* Streamlit is a **validation tool**, not a production system — use it to de-risk ideas before commissioning a full build

* The gap between AI prototype and stakeholder demo is a change in **inputs and outputs** — the AI logic does not change

## Resources

- [Streamlit Documentation](https://docs.streamlit.io/) — Full reference for all components
- [Streamlit API Reference](https://docs.streamlit.io/library/api-reference) — Quick lookup for widgets and layout
- [Streamlit Gallery](https://streamlit.io/gallery) — Explore what others have built with AI and data
- [Streamlit Community Cloud](https://streamlit.io/cloud) — Deploy your app to a public URL in minutes
- [30 Days of Streamlit](https://30days.streamlit.app/) — Hands-on self-paced learning challenge
- [Streamlit Cheat Sheet](https://docs.streamlit.io/library/cheatsheet) — Everything on one page
- [LLM Streamlit example apps](https://github.com/streamlit/llm-examples) — Starting points for LLM-powered demos
- Murray, D. (2023), *Creating Interactive Dashboards with Streamlit and Python*, Packt